In [1]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
max_seq_length = 4096 
load_in_4bit = True 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct", 
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
   
)

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4080 SUPER. Num GPUs = 1. Max memory: 15.55 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [      # training matrix
        "q_proj", "k_proj", "v_proj", "o_proj",     # attention layer
        "gate_proj", "up_proj", "down_proj",        # NLP layer
    ],
    lora_alpha = 16,        # scalar
    lora_dropout = 0,       # 
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 1313,
    use_rslora = False,     # if True: alpha/r -->alpha/sqrt(r) highrank stablizer 
)

Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [4]:
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    contexts     = examples["context"] 
    outputs      = examples["command"] 
    texts = []
    
    system_prompt = "You are a logic engine. Your output MUST follow this format:\n<think>\n[Your reasoning here]\n</think>\n<answer>\n[Your final concise answer here]\n</answer>"

    for instruction, context, output in zip(instructions, contexts, outputs):
        text = f"### System:\n{system_prompt}\n\n### Instruction:\n{instruction}\n\n### Context:\n{context}\n\n### Response:\n<think>\nZiel: {instruction}\nAnalysiere Kontext: {context}\nSyntax-Check: hyprctl/pacman Validierung.\nKonstruiere finalen Befehl.\n</think>\n<answer>\n{output}\n</answer>".strip()
        texts.append(text)
    return { "text" : texts, }

from datasets import load_dataset
dataset = load_dataset("jsonl", data_files="deine_daten.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

DatasetNotFoundError: Dataset 'jsonl' doesn't exist on the Hub or cannot be accessed.

In [5]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,

    args = TrainingArguments(
        output_dir = "outputs",
        
        # 1. TensorBoard Setup
        report_to = "tensorboard",     # Hier sagen wir: "Schick die Daten an TensorBoard"
        logging_dir = "outputs/logs",  # Wo die TensorBoard-Dateien landen
        logging_steps = 1,             # JEDEN Schritt loggen (damit der Graph live mitschreibt)
        
        # 2. Balken & Anzeige Setup
        disable_tqdm = False,          # Stellt sicher, dass der Fortschrittsbalken AN ist
        log_level = "info",            # Gibt dir mehr Details im Terminal
        
        # 3. Performance & Stabilität (wichtig für deine 4080)
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        learning_rate = 2e-4,
        max_steps = 60,                # Erstmal kurz testen
        optim = "adamw_8bit",          # VRAM sparen
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        
        # 4. Präzision
        bf16 = True,                   # Beste Wahl für die RTX 4080
    )
    )

NameError: name 'dataset' is not defined

In [ ]:
# 1. start training
trainer_stats = trainer.train()

# 2. save LoRA-Adapter
model.save_pretrained("arch_assistant_lora") 
tokenizer.save_pretrained("arch_assistant_lora")

print(f"Training abgeschlossen! Dauer: {trainer_stats.metrics['train_runtime']} Sekunden")

In [6]:
%load_ext tensorboard
%tensorboard --logdir outputs